In [4]:
from torchvision.transforms import v2
from torchvision import tv_tensors


from pathlib import Path
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import kagglehub
import datasets
import torch
from torchvision import transforms
import math
from datasets import load_dataset
# Download latest version
path = kagglehub.competition_download('airbus-ship-detection')

print("Path to competition files:", path)


/Users/ruslanomchenko/Documents/new_code_projects/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: /Users/ruslanomchenko/.cache/kagglehub/competitions/airbus-ship-detection


In [5]:
path_obj = Path(path)
train_segmentations = pd.read_csv(path_obj / "train_ship_segmentations_v2.csv")
obj = train_segmentations[~train_segmentations.EncodedPixels.isna()].iloc[1]
image = cv2.imread(path_obj / "train_v2" / obj.ImageId)

In [6]:
positive_part = train_segmentations.EncodedPixels.index[train_segmentations.EncodedPixels.notna()]
negative_part = train_segmentations.EncodedPixels.index[~train_segmentations.EncodedPixels.notna()]
np.random.choice(positive_part, size = (int(positive_part.shape[0]*0.8))),np.asarray(train_segmentations.index)
# training_paths = np.random.shuffle(np.concatenate((np.random.choice(positive_part, size = (positive_part.shape[0]*0.8)),
# np.random.choice(negative_part, size = (negative_part.shape[0]*0.8)))))


(array([ 70825, 182758, 110802, ..., 192742, 193007,  78409],
       shape=(65378,)),
 array([     0,      1,      2, ..., 231720, 231721, 231722],
       shape=(231723,)))

In [54]:
np.random.uniform(0, 1)

0.747242861786767

In [ ]:
class BackBoneDataset(datasets.GeneratorBasedBuilder):
    def __init__(self, datasetPaths, requiresMaskedLabels, 
                turn_prob = 0.5,
                prob_configs = {"hor_flip":0.4,
                                 "vertical_flip":0.3,
                                 "gaussian_blur":0.5},
                image_shape = {
                    "h":768,
                    "w":768,
                    "c":3
                }):
        
        self.datasetPaths = datasetPaths ## add random permutation
        self.requiresMaskedLabels = requiresMaskedLabels
        self.probabilities = prob_configs
        self.HWChannels = image_shape
        self.turnProb = turn_prob
        super().__init__()
        
        # create selecting positive and negative examples from the dataset
        positive_part = self.datasetPaths.EncodedPixels.index[self.datasetPaths.EncodedPixels.notna()]
        negative_part = self.datasetPaths.EncodedPixels.index[~self.datasetPaths.EncodedPixels.notna()]
        self.training_paths = np.concatenate((np.random.choice(positive_part, size = int(positive_part.shape[0]*0.8)),
        np.random.choice(negative_part, size = int(negative_part.shape[0]*0.8))))
        np.random.shuffle(self.training_paths)

        
        self.testing_paths = np.array( set(np.asarray(self.datasetPaths.index)).difference(set(self.training_paths)))


    def _info(self):
        return datasets.DatasetInfo(
            description="training dataset for cv task",
            features = datasets.Features({
            "path": datasets.Value("string"),
            "image": datasets.Array3D(shape=(self.HWChannels["c"],
                                             self.HWChannels["h"],
                                             self.HWChannels["w"]),
                                    dtype="float32"),
            "labels": datasets.Value("int8") if not self.requiresMaskedLabels else \
                                    datasets.Array3D(shape=(1,
                                             self.HWChannels["h"],
                                             self.HWChannels["w"]),
                                    dtype="float32")
            }))

    def _split_generators(self, dl_manager):

        return [
            datasets.SplitGenerator(name=datasets.Split.TRAIN,
                                    gen_kwargs = {
                                        "split": datasets.Split.TRAIN, 
                                        "indices": self.training_paths,
                                        "imageId_list":self.datasetPaths.loc[self.training_paths].ImageId,
                                        "EncodedPixels_list": self.datasetPaths.loc[self.training_paths].EncodedPixels
                                    }),
            datasets.SplitGenerator(name=datasets.Split.TEST,
                                    gen_kwargs = {
                                        "split": datasets.Split.TEST,
                                        "indices": self.testing_paths,
                                        "imageId_list":self.datasetPaths.loc[self.testing_paths].ImageId,
                                        "EncodedPixels_list": self.datasetPaths.loc[self.testing_paths].EncodedPixels
                                    }
                                    )
        ]
        

    def _generate_examples(self, split, indices, imageId_list, EncodedPixels_list):
        # yield and random seed
        # self.paths = random permute

        # create callback on epoch end(if streaming) -> after processing a certain amount of steps, to randomize training examples
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        #random.seed(worker_seed)
        def create_mask_with_gaussian(image, coords):
            image_ = np.zeros_like(image[:, :, 0]).astype(np.uint8)
            
            # coords_ = coords[coords[:, 0].argsort()]
            # coords_ = coords_[coords_[:, 1].argsort()]
            kernels = np.random.randint(5, 15)*np.ones(2).astype(int)*2+1
            image_ = cv2.fillPoly(image_, [coords], color=(255, 255, 255))
            
            image_ = cv2.GaussianBlur(image_.astype(float), ksize=kernels, sigmaX=kernels[0], sigmaY=kernels[1])
            return image_

        def bounding_box(item_):
            coords_ = item_.split(" ")
            #coords_ = np.array(coords_span[::2]).astype(int)
            #coords_.sort()
            item = [(int(c)//image.shape[1], int(c)%image.shape[1]) for c in coords_[::2]]
            spans = [int(sp) for sp in coords_[1::2]]
            item = item + [(item[i][0], item[i][1]+spans[i]) for i in range(len(item))]
            
            item = np.asarray(item)
            y = item[item[:, 1].argmin()].reshape(-1, 2)
            y_min = y[y[:, 0].argmin()]

            x = item[item[:, 0].argmin()].reshape(-1, 2)
            x_min = x[x[:, 1].argmax()]

            x = item[item[:, 1].argmax()].reshape(-1, 2)
            x_max = x[x[:, 0].argmin()]

            y = item[item[:, 0].argmax()].reshape(-1, 2)
            y_max = y[y[:, 1].argmax()]
            return np.array((y_min, x_min, x_max, y_max))
        for i in np.arange(imageId_list.shape[0]):
            label = EncodedPixels_list.iloc[i]
            image = cv2.imread(path_obj / "train_v2" / imageId_list.iloc[i])
            bounding_box_ = bounding_box(label) if not pd.isna(label) else None

            if self.requiresMaskedLabels:
                target = create_mask_with_gaussian(image, bounding_box_)
            else:
                target = int(not pd.isna(label))
            if split == 'train':
                if bounding_box_ is not None:
                    mass = np.mean(bounding_box_, axis=0)
                    if self.turnProb >= np.random.uniform(0, 1):
                        distances = np.asarray([[image.shape[0]//2,image.shape[1]], [image.shape[0]//2, 0], [0, image.shape[1]//2], [image.shape[0], image.shape[1]//2]])
                        #min_distance = np.linalg.norm(bounding_box_[:, :, None] - np.ones((4, *bounding_box_.shape))*bounding_box_, axis=-1)
                        degr = (-5+ np.random.random()*10)/57.29
                    
                        min_ = np.argmin(np.linalg.norm(np.transpose(np.ones((4, *bounding_box_.shape))*bounding_box_, axes = (1, 0, 2)) - np.ones((4, *bounding_box_.shape))*distances, axis=-1).flatten())
                        min_point = min_//4
                        min_side = min_%4
                        v1 = bounding_box_[min_point]-mass
                        v2_ = distances[min_side]-mass
                        theta = np.arccos(v1@v2_ / (np.linalg.norm(v1)*np.linalg.norm(v2_)))
                        catet2 = v1 * np.sin(theta)
                        max_theta = np.arctan2(np.linalg.norm(catet2), np.linalg.norm(v2_))
                        turn_angle = (theta - max_theta, degr)[np.argmin([abs(theta - max_theta), abs(degr)])]
                    else:turn_angle=0
                else: 

                    turn_angle = (-5+ np.random.random()*10)/57.29 if self.turnProb >= np.random.uniform(0, 1) else 0
                    mass = (self.HWChannels["h"]//2,self.HWChannels["w"]//2)
            
                #cv2.line(, [0,0], np.array(image.shape[:2])-1, color=(255,255,255))
            
                rotation_matrix = cv2.getRotationMatrix2D(mass, turn_angle*57.29, 1)
            
                image_turned = cv2.warpAffine(image.copy(), M = rotation_matrix, dsize = image.shape[:2])
                mask_turned = target if isinstance(target, int) else cv2.warpAffine(target.copy(), M = rotation_matrix, dsize = image.shape[:2])
                
                augmentation = v2.Compose([
                    v2.ColorJitter(
                        brightness=(0.7, 1.5),
                        contrast=(0.7, 1.5),
                        saturation=(0.5, 1.7),
                        hue=(-0.3, 0.3),
                    ),
                    v2.RandomHorizontalFlip(p=0.4),
                    v2.RandomVerticalFlip(p=0.3),
                    v2.RandomApply(
                        [v2.GaussianBlur(kernel_size=5, sigma=(0.1, 0.5))],
                        p=0.5,
                    ),
                ])
                #augmentation.

                image = tv_tensors.Image(
                    torch.tensor(image_turned)
                        .permute(2, 0, 1)
                        .float() / 255
                )

                mask = mask_turned if isinstance(mask_turned, int) else tv_tensors.Mask(
                    torch.tensor(mask_turned).float() / 255
                )
                image, target = augmentation(image, mask) if not isinstance(mask_turned, int) else augmentation(image), mask
                # target = mask_turned if isinstance(mask_turned, int) else \
                # augmentation(torch.tensor(mask_turned).float() / 255)
            else: 
                image = torch.tensor(image).permute(2, 0, 1).float() / 255
                target = target if isinstance(target, int) else \
                    torch.tensor(target).float() / 255
            yield indices[i], {
                "path":imageId_list.iloc[i],
                "image":image,
                "labels":target
            }



In [82]:
ds = BackBoneDataset(train_segmentations, False)

### TODO
### 1.Test of download and prepare with streaming, using 1. no mask 2.mask
### 2. Try to externally shuffle the entries in the dataset splits for the data to check if the randomness works(of the streaming order)
### 3. check if the augmentations are random enough(check whether pytorch and np seed is correct)
### 4. Write further stuff and prove that it loads correctly for next steps
### 5. Create Model class and Callbacks for randomization(from step 2), create collator for putting everything on gpu
### 6. Create train test loops for classification
ds = BackBoneDataset(train_segmentations[~train_segmentations.EncodedPixels.isna()], True)

### TODO
### 1.Test of download and prepare with streaming, using 1. no mask 2.mask
### 2. Try to externally shuffle the entries in the dataset splits for the data to check if the randomness works(of the streaming order)
### 3. check if the augmentations are random enough(check whether pytorch and np seed is correct)
### 4. Write further stuff and prove that it loads correctly for next steps
### 5. Create Model class and Callbacks for randomization(from step 2), create collator for putting everything on gpu
### 6. Create train test loops for classification

constructor
constructor


In [81]:
train_segmentations[~train_segmentations.EncodedPixels.isna()]

,ImageId,EncodedPixels
2,000155de5.jpg,264661 17 265429 33 266197 33 266965 33 267733...
3,000194a2d.jpg,360486 1 361252 4 362019 5 362785 8 363552 10 ...
4,000194a2d.jpg,51834 9 52602 9 53370 9 54138 9 54906 9 55674 ...
5,000194a2d.jpg,198320 10 199088 10 199856 10 200624 10 201392...
6,000194a2d.jpg,55683 1 56451 1 57219 1 57987 1 58755 1 59523 ...
...,...,...
231694,fff909871.jpg,441057 2 441825 4 442592 7 443360 9 444128 11 ...
231709,fffcb6464.jpg,222782 4 223550 10 224318 14 225086 14 225854 ...
231712,fffd924fb.jpg,110011 5 110779 5 111547 5 112315 5 113083 5 1...
231713,fffd924fb.jpg,198440 8 199208 8 199976 8 200744 8 201512 8 2...


In [83]:
train_ds = ds.as_streaming_dataset(split="train")
test_ds = ds.as_streaming_dataset(split="test")

In [97]:
for example in train_ds:
    res = example
    break

[[426 362]
 [420 377]
 [440 387]
 [447 370]] [[[83 71 23]
  [81 69 21]
  [79 67 19]
  ...
  [80 68 20]
  [83 71 23]
  [84 72 24]]

 [[83 71 23]
  [81 69 21]
  [79 67 19]
  ...
  [81 69 21]
  [83 71 23]
  [84 72 24]]

 [[83 71 23]
  [81 69 21]
  [80 68 20]
  ...
  [81 69 21]
  [84 72 24]
  [85 73 25]]

 ...

 [[82 73 23]
  [85 76 26]
  [86 77 27]
  ...
  [83 74 24]
  [81 72 22]
  [79 70 20]]

 [[82 73 23]
  [84 75 25]
  [85 76 26]
  ...
  [81 72 22]
  [78 69 19]
  [76 67 17]]

 [[82 73 23]
  [84 75 25]
  [85 76 26]
  ...
  [80 71 21]
  [76 67 17]
  [74 65 15]]]


In [99]:
from torchvision.transforms.functional import to_pil_image
img = example["image"][0].copy()
#img*=example["image"][1]
img_tensor = torch.Tensor(img)  # e.g. shape [C, H, W]
img_pil = to_pil_image(img_tensor)

img_pil.show()

In [92]:
example['image'][1]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(768, 768), dtype=float32)

In [88]:
example['image'][0]

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]],
      shape=(3, 768, 768), dtype=float32)

In [20]:
v2

<module 'torchvision.transforms.v2' from '/Users/ruslanomchenko/Documents/new_code_projects/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/__init__.py'>

In [66]:
example

{'path': 'bdf204e86.jpg',
 'image': array([[[0.54650086, 0.48763362, 0.4640866 , ..., 0.6488867 ,
          0.66066015, 0.66654694],
         [0.51706713, 0.4640866 , 0.43465286, ..., 0.63711333,
          0.6488867 , 0.6547734 ],
         [0.46997336, 0.43465286, 0.39933252, ..., 0.619453  ,
          0.63122654, 0.64299995],
         ...,
         [0.6309141 , 0.6309141 , 0.6309141 , ..., 0.5014059 ,
          0.5426129 , 0.56615996],
         [0.64268756, 0.6309141 , 0.6250274 , ..., 0.4837457 ,
          0.5072926 , 0.5249528 ],
         [0.66034776, 0.6368008 , 0.6309141 , ..., 0.46019867,
          0.47197223, 0.47785896]],
 
        [[0.58350956, 0.5246422 , 0.5010953 , ..., 0.6766402 ,
          0.6884137 , 0.6943004 ],
         [0.5540758 , 0.5010953 , 0.47166163, ..., 0.66486675,
          0.6766402 , 0.6825269 ],
         [0.5069821 , 0.47166163, 0.43634126, ..., 0.6472065 ,
          0.65897995, 0.6707534 ],
         ...,
         [0.6355418 , 0.6355418 , 0.6355418 , ..., 0